# 🧠 Sentence-BERT (SBERT) Vector Ingestion & Embedding Pipeline

Welcome to the **SBERT Embedding Pipeline**! This notebook defines a high-performance, local embedding pipeline utilizing SentenceTransformers (SBERT). It is designed to work in synergy with the FastAPI backend.

### ✨ Key Capabilities
1. **Local & Private Embeddings**: Uses high-performance local SBERT models (`all-MiniLM-L6-v2`) without relying on external API keys.
2. **Dual-Mode Pipeline**: Can receive a string input from the backend:
   - If the input is a **PDF filename** from the `uploaded_pdfs` directory, it extracts, chunks, embeds, and indexes it into **ChromaDB**.
   - If the input is a **text query/string**, it returns the raw SBERT embedding vector immediately.

In [4]:
!pip install langchain-community langchain-chroma sentence-transformers

  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached langchain_classic-1.0.7-py3-none-any.whl.metadata (5.1 kB)
  Using cached langchain_core-1.4.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached langsmith-0.8.5-py3-none-any.whl.metadata (15 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached langchain_protocol-0.0.15-py3-none-any.whl.metadata (2.4 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using ca

In [5]:
import os
import glob#
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer

print("✅ Libraries imported successfully!")

## ⚙️ Configuration & Model Selection
Here we select the **SBERT model** to generate the embeddings. The default is `all-MiniLM-L6-v2`, which is exceptionally fast, lightweight, and accurate.

In [ ]:
# Configuration Constants
UPLOAD_DIR = "uploaded_pdfs"
DB_DIR = os.path.join("db", "doc_embeddings_sbert")

# Choose your SBERT model here
MODEL_NAME = "all-MiniLM-L6-v2"

print(f" Loading local SBERT model '{MODEL_NAME}'...")
embeddings = HuggingFaceEmbeddings(model_name=MODEL_NAME)
print("Model loaded successfully and ready to use!")

/var/folders/yn/5rpzxtkx20x54fhnmf__zr_h0000gn/T/ipykernel_17503/1558060290.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


ModuleNotFoundError: No module named 'langchain'

## 🚀 Main Embedding Pipeline
This is the core function called by the backend or notebook execution. It determines whether to embed a PDF file or a general text query.

In [ ]:
def generate_vector_embeddings(input_str: str) -> dict:
    """
    Generate vector embeddings using SBERT.
    
    Parameters:
        input_str (str): Either a PDF filename in `uploaded_pdfs/` or raw query text.
        
    Returns:
        dict: Success details and metadata or embedding vectors.
    """
    # Resolve PDF path
    pdf_path = os.path.join(UPLOAD_DIR, input_str)
    
    # Handle missing .pdf extension automatically
    if not os.path.exists(pdf_path) and not input_str.endswith(".pdf"):
        pdf_path_with_ext = pdf_path + ".pdf"
        if os.path.exists(pdf_path_with_ext):
            pdf_path = pdf_path_with_ext
            
    # Case 1: The input is a PDF file in uploaded_pdfs
    if os.path.exists(pdf_path) and os.path.isfile(pdf_path) and pdf_path.endswith(".pdf"):
        print(f" Found PDF file: {pdf_path}. Commencing extraction & embedding...")
        
        # 1. Load and parse the PDF file
        loader = PyPDFLoader(pdf_path)
        documents = loader.load()
        
        # 2. Split PDF text into manageable chunks
        splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
        docs = splitter.split_documents(documents)
        print(f" Split PDF into {len(docs)} document chunks.")
        
        # 3. Create Chroma Vector DB and store embeddings locally
        print(f" Storing embeddings in ChromaDB at '{DB_DIR}'...")
        db = Chroma.from_documents(
            docs,
            embeddings,
            persist_directory=DB_DIR
        )
        print("🎉 Successfully populated Vector DB!")
        
        return {
            "status": "success",
            "type": "pdf_ingestion",
            "filename": os.path.basename(pdf_path),
            "num_chunks": len(docs),
            "db_directory": DB_DIR
        }
        
    # Case 2: The input is a raw text query/string
    else:
        print(" Input is not a PDF file. Generating vector embedding for text query...")
        vector = embeddings.embed_query(input_str)
        print(f"✨ Successfully generated vector embedding of dimension {len(vector)}.")
        
        return {
            "status": "success",
            "type": "text_query",
            "vector_length": len(vector),
            "vector": vector
        }

## 🧪 Local Verification / Demonstration
Let's run a test query and check if the database parses a PDF.

In [ ]:
# Example 1: Embedding a text query
query_test = "How to integrate Sentence Transformers with FastAPI?"
result_query = generate_vector_embeddings(query_test)
print(f"Vector dimension: {result_query['vector_length']}")
print(f" Sample vector values: {result_query['vector'][:5]}...")

In [ ]:
# Example 2: Ingesting an uploaded PDF
pdf_files = glob.glob(os.path.join(UPLOAD_DIR, "*.pdf"))
if pdf_files:
    sample_pdf = os.path.basename(pdf_files[0])
    print(f"Found sample PDF: {sample_pdf}")
    result_pdf = generate_vector_embeddings(sample_pdf)
    print(result_pdf)
else:
    print("ℹNo PDFs found in 'uploaded_pdfs/' directory to run demonstration.")